# Topo-Brain Pipeline Controller

**Simple notebook to run and monitor the existing training/inference scripts.**

This notebook uses the production-ready scripts in `scripts/` directory:
- `train_diffusion.py` - Full training with all features
- `sample_diffusion.py` - Quick patch inference
- `evaluate_full_volume.py` - Full-volume evaluation with tiling
- `find_best_checkpoint.py` - Find best checkpoint

**Why use this notebook?**
- Easy configuration without command-line arguments
- Live monitoring of training progress
- Visualize results interactively
- Quick access to all pipeline steps

---
## Setup & Configuration

In [ ]:
import os
import sys
import subprocess
import time
import yaml
from pathlib import Path
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from IPython.display import display, HTML, clear_output
import pandas as pd
from tqdm.notebook import tqdm

# Project root
PROJECT_ROOT = Path(os.getcwd()).parent if Path(os.getcwd()).name == 'notebooks' else Path(os.getcwd())
os.chdir(PROJECT_ROOT)

print(f"✓ Project root: {PROJECT_ROOT}")
print(f"✓ Scripts directory: {PROJECT_ROOT / 'scripts'}")
print(f"✓ Config directory: {PROJECT_ROOT / 'configs'}")

In [ ]:
# ============================================================
# CONFIGURATION - Edit these paths for your setup
# ============================================================

CONFIG = {
    # Paths
    'config_file': PROJECT_ROOT / 'configs' / 'train_diffusion.yaml',
    'pairs_csv': PROJECT_ROOT / 'pairs_new.csv',
    'data_root': None,  # Set to your data directory or None if CSV has absolute paths
    'masks_root': None,  # Set to your masks directory or None
    'output_dir': PROJECT_ROOT / 'output',
    
    # Training
    'resume_checkpoint': None,  # Path to checkpoint or None for fresh start
    'use_wandb': False,  # Enable Weights & Biases logging
    'wandb_project': 'topobrain',
    'wandb_entity': None,
    
    # Evaluation
    'test_subject': 'sub-01',  # Subject to evaluate
    'eval_checkpoint': None,  # Will use latest if None
}

# Create output directory
CONFIG['output_dir'].mkdir(parents=True, exist_ok=True)

# Load and display config file
with open(CONFIG['config_file']) as f:
    train_config = yaml.safe_load(f)

print("Current Configuration:")
print(f"  Config file: {CONFIG['config_file']}")
print(f"  Pairs CSV: {CONFIG['pairs_csv']}")
print(f"  Output dir: {CONFIG['output_dir']}")
print(f"  Batch size: {train_config['dataset']['batch_size']}")
print(f"  Total iterations: {train_config['training']['n_iters']:,}")
print(f"  Use W&B: {CONFIG['use_wandb']}")

---
## 1. Data Verification

Check that all data files exist before starting training.

In [ ]:
import csv

# Load pairs CSV
pairs = []
with open(CONFIG['pairs_csv']) as f:
    reader = csv.DictReader(f)
    pairs = list(reader)

print(f"Found {len(pairs)} subject pairs\n")

# Verify files
missing = []
for p in pairs:
    subj = p['subject']
    
    # Build full paths
    inp = Path(CONFIG['data_root'] or '') / p['input_3t']
    tgt = Path(CONFIG['data_root'] or '') / p['target_7t']
    seg = Path(CONFIG['masks_root'] or CONFIG['data_root'] or '') / p.get('seg', '') if p.get('seg') else None
    
    inp_ok = inp.exists()
    tgt_ok = tgt.exists()
    seg_ok = seg.exists() if seg else False
    
    status = '✓' if (inp_ok and tgt_ok) else '✗'
    print(f"{status} {subj}: input={inp_ok}, target={tgt_ok}, seg={seg_ok}")
    
    if not (inp_ok and tgt_ok):
        missing.append(subj)

if missing:
    print(f"\n⚠ WARNING: {len(missing)} subjects have missing files: {missing}")
else:
    print(f"\n✓ All files verified successfully!")

---
## 2. Training

Run the training script with configured parameters.

In [ ]:
def build_train_command():
    """Build training command from CONFIG."""
    cmd = [
        'python', 'scripts/train_diffusion.py',
        '--config', str(CONFIG['config_file']),
        '--output', str(CONFIG['output_dir']),
    ]
    
    if CONFIG['data_root']:
        cmd.extend(['--data-root', str(CONFIG['data_root'])])
    
    if CONFIG['masks_root']:
        cmd.extend(['--masks-root', str(CONFIG['masks_root'])])
    
    if CONFIG['resume_checkpoint']:
        cmd.extend(['--resume', str(CONFIG['resume_checkpoint'])])
    
    if CONFIG['use_wandb']:
        cmd.append('--use-wandb')
        cmd.extend(['--wandb-project', CONFIG['wandb_project']])
        if CONFIG['wandb_entity']:
            cmd.extend(['--wandb-entity', CONFIG['wandb_entity']])
    
    return cmd

# Show command
train_cmd = build_train_command()
print("Training command:")
print(' '.join(train_cmd))
print("\nReady to start training!")

In [ ]:
# ============================================================
# Option 1: Run training in background (non-blocking)
# ============================================================
# Uncomment to use:

# print("Starting training in background...")
# log_file = CONFIG['output_dir'] / 'training.log'
# with open(log_file, 'w') as f:
#     process = subprocess.Popen(
#         train_cmd,
#         stdout=f,
#         stderr=subprocess.STDOUT,
#         cwd=PROJECT_ROOT
#     )
# 
# print(f"✓ Training started (PID: {process.pid})")
# print(f"  Log file: {log_file}")
# print(f"  Monitor: tail -f {log_file}")
# print(f"\nYou can continue working in other cells while training runs.")

In [ ]:
# ============================================================
# Option 2: Run training with live output (blocking)
# ============================================================
# Uncomment to use:

# print("Starting training with live output...\n")
# subprocess.run(train_cmd, cwd=PROJECT_ROOT)

In [ ]:
# ============================================================
# Monitor Training Progress
# ============================================================

def monitor_training(log_file, refresh_interval=5):
    """Monitor training progress from log file."""
    log_file = Path(log_file)
    if not log_file.exists():
        print(f"Log file not found: {log_file}")
        return
    
    print(f"Monitoring: {log_file}")
    print("Press 'Interrupt Kernel' to stop monitoring\n")
    
    last_size = 0
    try:
        while True:
            current_size = log_file.stat().st_size
            if current_size > last_size:
                with open(log_file) as f:
                    f.seek(last_size)
                    new_lines = f.read()
                    print(new_lines, end='')
                last_size = current_size
            time.sleep(refresh_interval)
    except KeyboardInterrupt:
        print("\n✓ Stopped monitoring")

# Uncomment to monitor:
# monitor_training(CONFIG['output_dir'] / 'training.log')

---
## 3. Checkpoint Management

List and analyze saved checkpoints.

In [ ]:
# List all checkpoints
ckpt_dirs = sorted(CONFIG['output_dir'].glob('checkpoint_*/'))
ckpt_files = [d / f"checkpoint_{d.name.split('_')[1]}.pt" for d in ckpt_dirs if d.is_dir()]
ckpt_files = [f for f in ckpt_files if f.exists()]

# Add latest if exists
latest = CONFIG['output_dir'] / 'checkpoint_latest.pt'
if latest.exists() and latest not in ckpt_files:
    ckpt_files.insert(0, latest)

if ckpt_files:
    print(f"Found {len(ckpt_files)} checkpoints:\n")
    
    ckpt_info = []
    for ckpt in ckpt_files:
        size_mb = ckpt.stat().st_size / 1e6
        mtime = time.ctime(ckpt.stat().st_mtime)
        
        # Try to load step info
        try:
            import torch
            data = torch.load(ckpt, map_location='cpu')
            step = data.get('step', '?')
            del data
        except:
            step = '?'
        
        ckpt_info.append({
            'File': ckpt.name,
            'Step': step,
            'Size (MB)': f"{size_mb:.1f}",
            'Modified': mtime,
        })
    
    df = pd.DataFrame(ckpt_info)
    display(df)
    
    print(f"\nLatest checkpoint: {latest if latest.exists() else 'None'}")
else:
    print("No checkpoints found. Run training first.")

In [ ]:
# Find best checkpoint using the script
def find_best_checkpoint():
    """Run find_best_checkpoint.py script."""
    cmd = ['python', 'scripts/find_best_checkpoint.py', '--checkpoint-dir', str(CONFIG['output_dir'])]
    
    if CONFIG['data_root']:
        cmd.extend(['--data-root', str(CONFIG['data_root'])])
    
    print("Finding best checkpoint...\n")
    result = subprocess.run(cmd, cwd=PROJECT_ROOT, capture_output=True, text=True)
    print(result.stdout)
    if result.stderr:
        print("Errors:", result.stderr)

# Uncomment to find best checkpoint:
# find_best_checkpoint()

---
## 4. Quick Patch Inference

Fast inference on center 64³ crop using `sample_diffusion.py`.

In [ ]:
# Select checkpoint for inference
INFERENCE_CKPT = CONFIG['eval_checkpoint'] or (CONFIG['output_dir'] / 'checkpoint_latest.pt')

if not INFERENCE_CKPT.exists():
    print(f"⚠ Checkpoint not found: {INFERENCE_CKPT}")
    print("Set CONFIG['eval_checkpoint'] to a valid checkpoint path.")
else:
    print(f"✓ Using checkpoint: {INFERENCE_CKPT}")

In [ ]:
# Run quick patch inference
def run_patch_inference(subject_id):
    """Run sample_diffusion.py on a subject."""
    # Find subject in pairs
    pair = next((p for p in pairs if p['subject'] == subject_id), None)
    if not pair:
        print(f"Subject {subject_id} not found in pairs CSV")
        return
    
    # Build paths
    inp = Path(CONFIG['data_root'] or '') / pair['input_3t']
    tgt = Path(CONFIG['data_root'] or '') / pair['target_7t']
    out_dir = CONFIG['output_dir'] / 'results' / subject_id / 'patch'
    out_dir.mkdir(parents=True, exist_ok=True)
    out_file = out_dir / 'predicted_7T.nii.gz'
    
    # Build command
    cmd = [
        'python', 'scripts/sample_diffusion.py',
        '--checkpoint', str(INFERENCE_CKPT),
        '--input', str(inp),
        '--target', str(tgt),
        '--output', str(out_file),
    ]
    
    print(f"Running patch inference on {subject_id}...\n")
    result = subprocess.run(cmd, cwd=PROJECT_ROOT, capture_output=True, text=True)
    print(result.stdout)
    if result.stderr:
        print("Errors:", result.stderr)
    
    return out_dir

# Run on test subject
patch_results = run_patch_inference(CONFIG['test_subject'])

In [ ]:
# Visualize patch inference results
if patch_results:
    pred_file = patch_results / 'predicted_7T.nii.gz'
    vis_file = patch_results.parent / (patch_results.name + '.png')
    
    if vis_file.exists():
        from PIL import Image
        img = Image.open(vis_file)
        plt.figure(figsize=(20, 8))
        plt.imshow(img)
        plt.axis('off')
        plt.title(f'Patch Inference Results - {CONFIG["test_subject"]}', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()
    
    if pred_file.exists():
        print(f"\n✓ Prediction saved: {pred_file}")

---
## 5. Full-Volume Evaluation

Tiled inference on complete volumes using `evaluate_full_volume.py`.

In [ ]:
def run_full_volume_eval(subject_id, overlap=32):
    """Run evaluate_full_volume.py on a subject."""
    out_dir = CONFIG['output_dir'] / 'results' / subject_id / 'full_volume'
    out_dir.mkdir(parents=True, exist_ok=True)
    
    # Build command
    cmd = [
        'python', 'scripts/evaluate_full_volume.py',
        '--checkpoint', str(INFERENCE_CKPT),
        '--subject', subject_id,
        '--overlap', str(overlap),
        '--output_dir', str(out_dir),
    ]
    
    if CONFIG['data_root']:
        cmd.extend(['--data-root', str(CONFIG['data_root'])])
    
    if CONFIG['masks_root']:
        cmd.extend(['--masks-root', str(CONFIG['masks_root'])])
    
    if CONFIG['pairs_csv']:
        cmd.extend(['--pairs_csv', str(CONFIG['pairs_csv'])])
    
    print(f"Running full-volume evaluation on {subject_id}...")
    print(f"Output: {out_dir}\n")
    
    result = subprocess.run(cmd, cwd=PROJECT_ROOT, capture_output=True, text=True)
    print(result.stdout)
    if result.stderr:
        print("Errors:", result.stderr)
    
    return out_dir

# Run on test subject
full_results = run_full_volume_eval(CONFIG['test_subject'])

In [ ]:
# Visualize full-volume results
if full_results:
    # Show all generated PNG visualizations
    vis_files = sorted(full_results.glob('eval_*.png'))
    
    if vis_files:
        print(f"Found {len(vis_files)} visualization files:\n")
        
        for vis_file in vis_files[:9]:  # Show first 9
            from PIL import Image
            img = Image.open(vis_file)
            plt.figure(figsize=(20, 6))
            plt.imshow(img)
            plt.axis('off')
            plt.title(vis_file.name, fontsize=12)
            plt.tight_layout()
            plt.show()
    
    # Check for NIfTI outputs
    pred_nii = full_results / 'predicted_7T.nii.gz'
    seg_nii = full_results / 'predicted_seg.nii.gz'
    
    if pred_nii.exists():
        print(f"\n✓ Full-volume prediction: {pred_nii}")
    if seg_nii.exists():
        print(f"✓ Segmentation: {seg_nii}")

---
## 6. Evaluate All Checkpoints

Compare metrics across multiple checkpoints.

In [ ]:
def evaluate_all_checkpoints():
    """Run evaluate_all_checkpoints.py script."""
    cmd = [
        'python', 'scripts/evaluate_all_checkpoints.py',
        '--checkpoint-dir', str(CONFIG['output_dir']),
        '--output-dir', str(CONFIG['output_dir'] / 'checkpoint_comparison'),
    ]
    
    if CONFIG['data_root']:
        cmd.extend(['--data-root', str(CONFIG['data_root'])])
    
    if CONFIG['masks_root']:
        cmd.extend(['--masks-root', str(CONFIG['masks_root'])])
    
    print("Evaluating all checkpoints...\n")
    result = subprocess.run(cmd, cwd=PROJECT_ROOT, capture_output=True, text=True)
    print(result.stdout)
    if result.stderr:
        print("Errors:", result.stderr)

# Uncomment to evaluate all:
# evaluate_all_checkpoints()

---
## 7. Batch Evaluation (All Subjects)

Evaluate all subjects and aggregate metrics.

In [ ]:
def evaluate_all_subjects():
    """Run full-volume evaluation on all subjects."""
    results = []
    
    for pair in tqdm(pairs, desc="Evaluating subjects"):
        subj = pair['subject']
        
        try:
            out_dir = run_full_volume_eval(subj, overlap=32)
            results.append({'subject': subj, 'status': 'success', 'output': str(out_dir)})
        except Exception as e:
            print(f"\n⚠ Failed for {subj}: {e}")
            results.append({'subject': subj, 'status': 'failed', 'error': str(e)})
    
    # Save summary
    summary_df = pd.DataFrame(results)
    summary_file = CONFIG['output_dir'] / 'results' / 'evaluation_summary.csv'
    summary_df.to_csv(summary_file, index=False)
    
    print(f"\n✓ Evaluation complete. Summary saved to: {summary_file}")
    display(summary_df)

# Uncomment to evaluate all subjects:
# evaluate_all_subjects()

---
## 8. Load and Analyze Results

Load saved results and create summary visualizations.

In [ ]:
# Aggregate metrics from all subject evaluations
def collect_metrics():
    """Collect metrics from all evaluated subjects."""
    results_dir = CONFIG['output_dir'] / 'results'
    metrics_list = []
    
    for subj_dir in results_dir.glob('sub-*/'):
        subj = subj_dir.name
        
        # Check for full_volume results
        full_vol_dir = subj_dir / 'full_volume'
        if not full_vol_dir.exists():
            continue
        
        # Look for metrics in stdout/log (you may need to parse these)
        # For now, we'll note that metrics are printed
        metrics_list.append({'subject': subj, 'evaluated': True})
    
    if metrics_list:
        df = pd.DataFrame(metrics_list)
        print(f"Found results for {len(metrics_list)} subjects:")
        display(df)
    else:
        print("No evaluation results found. Run evaluations first.")

collect_metrics()

---
## Summary

This notebook provides an easy interface to the production scripts:

| Section | Script Used | Purpose |
|---------|-------------|--------|
| 1. Data Verification | - | Check files exist |
| 2. Training | `train_diffusion.py` | Full curriculum training |
| 3. Checkpoints | - | List and manage checkpoints |
| 4. Patch Inference | `sample_diffusion.py` | Quick 64³ inference |
| 5. Full Volume | `evaluate_full_volume.py` | Tiled inference + metrics |
| 6. All Checkpoints | `evaluate_all_checkpoints.py` | Compare checkpoints |
| 7. Batch Eval | `evaluate_full_volume.py` | All subjects |
| 8. Analysis | - | Aggregate and visualize |

**The scripts do all the heavy lifting - this notebook just makes them easier to use!**